# 데이터 로드

- annotation JSON 하나 = 이미지 1장 × 알약 1개입니다. 같은 이미지의 알약이 약 코드별 폴더에 흩어져 있어서 `file_name` 기준으로 합칩니다.
- 경로: `data/train_annotations/<조합>_json/<약코드>/<이미지이름>.json`
- bbox는 COCO 형식 `[x, y, w, h]`(왼쪽 위 기준, 픽셀)입니다.

In [1]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

# 노트북은 notebooks/에서 실행되므로 프로젝트 루트를 import 경로에 추가한다.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import PROJECT_ROOT  # noqa: E402

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_IMG_DIR = DATA_DIR / "train_images"
TRAIN_ANN_DIR = DATA_DIR / "train_annotations"
TEST_IMG_DIR = DATA_DIR / "test_images"

In [2]:
def load_annotations(ann_dir: Path, img_dir: Path) -> pd.DataFrame:
    """annotation JSON을 모두 읽어 bbox 한 개당 한 행인 DataFrame을 만든다."""
    rows = []
    for json_path in sorted(ann_dir.rglob("*.json")):
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        img = data["images"][0]
        categories = {c["id"]: c["name"] for c in data["categories"]}
        for ann in data["annotations"]:
            x, y, w, h = ann["bbox"]
            rows.append(
                {
                    **img,
                    "image_path": img_dir / img["file_name"],
                    "json_path": json_path,
                    "ann_id": ann["id"],
                    "category_id": ann["category_id"],
                    "category_name": categories.get(ann["category_id"]),
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "area": ann["area"],
                }
            )
    return pd.DataFrame(rows).rename(columns={"id": "image_id"})


ann_df = load_annotations(TRAIN_ANN_DIR, TRAIN_IMG_DIR)
ann_df.head()

,file_name,width,height,imgfile,drug_N,drug_S,back_color,drug_dir,light_color,camera_la,...,image_path,json_path,ann_id,category_id,category_name,x,y,w,h,area
0,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,K-001900,정상알약,연회색 배경,앞면,주백색,70,...,/Users/yundong-ug/dev/pillbbox/data/train_imag...,/Users/yundong-ug/dev/pillbbox/data/train_anno...,133,1900,보령부스파정 5mg,644,845,189,190,35910
1,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,K-001900,정상알약,연회색 배경,앞면,주백색,75,...,/Users/yundong-ug/dev/pillbbox/data/train_imag...,/Users/yundong-ug/dev/pillbbox/data/train_anno...,134,1900,보령부스파정 5mg,148,241,181,181,32761
2,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,K-001900,정상알약,연회색 배경,앞면,주백색,90,...,/Users/yundong-ug/dev/pillbbox/data/train_imag...,/Users/yundong-ug/dev/pillbbox/data/train_anno...,135,1900,보령부스파정 5mg,167,248,184,182,33488
3,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,K-016548,정상알약,연회색 배경,앞면,주백색,70,...,/Users/yundong-ug/dev/pillbbox/data/train_imag...,/Users/yundong-ug/dev/pillbbox/data/train_anno...,136,16548,가바토파정 100mg,144,799,239,239,57121
4,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,K-016548,정상알약,연회색 배경,앞면,주백색,75,...,/Users/yundong-ug/dev/pillbbox/data/train_imag...,/Users/yundong-ug/dev/pillbbox/data/train_anno...,137,16548,가바토파정 100mg,578,222,233,226,52658


In [4]:
# 이미지 한 장당 한 행. 파일 이름의 약 코드 수와 실제 bbox 수를 함께 둔다.
img_df = (
    ann_df.groupby("file_name")
    .agg(
        image_id=("image_id", "first"),
        image_path=("image_path", "first"),
        width=("width", "first"),
        height=("height", "first"),
        n_boxes=("ann_id", "size"),
    )
    .reset_index()
)
img_df["n_codes_in_name"] = img_df["file_name"].str.split("_").str[0].str.count("-")

test_paths = sorted(TEST_IMG_DIR.glob("*.png"), key=lambda p: int(p.stem))

n_classes = ann_df["category_id"].nunique()
disk_names = {p.name for p in TRAIN_IMG_DIR.glob("*.png")}
n_no_ann = len(disk_names - set(img_df["file_name"]))
n_missing = (img_df["n_boxes"] < img_df["n_codes_in_name"]).sum()

display(img_df.head(5))
print(f"train 이미지: {len(img_df)}장, bbox: {len(ann_df)}개, 클래스: {n_classes}개")
print(f"test 이미지: {len(test_paths)}장")
print(f"annotation 없는 train 이미지: {n_no_ann}장")
print(f"파일 이름보다 bbox가 적은 이미지: {n_missing}장")

,file_name,image_id,image_path,width,height,n_boxes,n_codes_in_name
0,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,34,/Users/yundong-ug/dev/pillbbox/data/train_imag...,976,1280,4,4
1,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,35,/Users/yundong-ug/dev/pillbbox/data/train_imag...,976,1280,4,4
2,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,36,/Users/yundong-ug/dev/pillbbox/data/train_imag...,976,1280,4,4
3,K-001900-016548-019607-033009_0_2_0_2_70_000_2...,1228,/Users/yundong-ug/dev/pillbbox/data/train_imag...,976,1280,4,4
4,K-001900-016548-019607-033009_0_2_0_2_75_000_2...,1230,/Users/yundong-ug/dev/pillbbox/data/train_imag...,976,1280,4,4


train 이미지: 232장, bbox: 763개, 클래스: 56개
test 이미지: 842장
annotation 없는 train 이미지: 0장
파일 이름보다 bbox가 적은 이미지: 8장


In [ ]:
def show_image(file_name: str) -> None:
    """train 이미지 한 장에 bbox와 category_id를 그려서 보여준다."""
    rows = ann_df[ann_df["file_name"] == file_name]
    img = cv2.cvtColor(cv2.imread(str(rows["image_path"].iloc[0])), cv2.COLOR_BGR2RGB)
    for r in rows.itertuples():
        cv2.rectangle(img, (r.x, r.y), (r.x + r.w, r.y + r.h), (255, 0, 0), 3)
        cv2.putText(
            img, str(r.category_id), (r.x, r.y - 8), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3
        )
    plt.figure(figsize=(6, 8))
    plt.imshow(img)
    plt.title(file_name, fontsize=9)
    plt.axis("off")
    plt.show()


show_image(img_df["file_name"].iloc[0])

Json 탐색 완료